## RQ3: Can EVT-based models provide better risk assessment for extreme events in the cryptocurrency market?

### Imports

In [ ]:
import os
import re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from collections import Counter
from scipy import stats
from scipy.stats import norm, t, genpareto, laplace, kurtosis, skew, gaussian_kde, ttest_ind
from scipy.stats import t as student_t 
from statsmodels.graphics.gofplots import qqplot
from arch import arch_model
from itertools import groupby
from operator import itemgetter
from openpyxl import load_workbook
from pandas.tseries.offsets import DateOffset
from scipy.optimize import minimize
from scipy.stats import gaussian_kde

# Importing pyextreme packages
from pyextremes import get_extremes, get_model, EVA, plot_mean_residual_life, plot_parameter_stability, plot_return_value_stability, plot_threshold_stability, get_extremes, get_return_periods
from pyextremes.plotting import plot_extremes
from pyextremes import plot_parameter_stability
%matplotlib inline

### Data

In [ ]:
def load_and_prepare_assets(asset_configs, base_path):
    """
    Load Excel files for multiple assets, set the correct index, and compute log and simple returns.


    Parameters:
        asset_configs (dict): Dictionary specifying for each asset:
            - file: Excel file name
            - date_col: name of the column to use as datetime index
            - price_col: name of the column containing the price to use
        base_path (str): Path prefix to locate all Excel files

    Returns:
        dict: Dictionary of processed pandas DataFrames with log and simple returns
    """
    asset_data = {}

    for asset_name, config in asset_configs.items():
        file_path = base_path + config['file']
        df = pd.read_excel(file_path)

        # Set datetime index
        df[config['date_col']] = pd.to_datetime(df[config['date_col']])
        df.set_index(config['date_col'], inplace=True)
        df.sort_index(inplace=True)

        # Compute returns
        df['Log_Returns'] = np.log(df[config['price_col']] / df[config['price_col']].shift(1))
        df['Simple_Returns'] = df[config['price_col']].pct_change()
        df.dropna(inplace=True)

        asset_data[asset_name] = df

    return asset_data

In [ ]:
def build_evt_dataset(
    assets, 
    return_type='Log_Returns', 
    quantile=0.95, 
    start_date='2010-01-01', 
    end_date='2025-05-09', 
    use_requested_start_date=False,
    align_dates=False
):
    """
    Aligns a dictionary of asset return series to a common time frame (optional), 
    recalculates thresholds, and supports flexible start date logic.

    Parameters:
        assets (dict): e.g. {'BTC': {'Log_Returns': pd.Series, 'Threshold': float}, ...}
        return_type (str): Column name of return series to use (e.g., 'Log_Returns')
        quantile (float): Tail quantile for EVT threshold
        start_date (str): Requested start date (used if use_requested_start_date=True)
        end_date (str): End date for analysis
        use_requested_start_date (bool): 
            - True → try using start_date for each asset (fallback if too early)
            - False → use the latest common start across all assets
        align_dates (bool):
            - True → intersect dates across all assets
            - False → keep assets independently trimmed

    Returns:
        dict: Aligned (or independently trimmed) asset dictionary with thresholds
    """
    aligned_assets = {}
    effective_start_dates = {}

    # Step 0: Establish effective global start date if needed
    if not use_requested_start_date:
        all_starts = [data[return_type].index.min() for data in assets.values()]
        start_date = max(all_starts)
        print(f"Using maximum common start date across all assets: {start_date.strftime('%Y-%m-%d')}")

    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    # Step 1: Trim individual assets based on their own available range
    for name, data in assets.items():
        series = data[return_type]
        asset_start = series.index.min()

        if use_requested_start_date and start_date < asset_start:
            print(f"WARNING: {name} has no data before {asset_start.strftime('%Y-%m-%d')}. Using that instead.")
            effective_start = asset_start
        else:
            effective_start = max(start_date, asset_start)

        trimmed_returns = series[effective_start:end_date]
        threshold = np.quantile(trimmed_returns, quantile)

        aligned_assets[name] = {
            return_type: trimmed_returns,
            'Threshold': threshold
        }
        effective_start_dates[name] = effective_start

    # Step 2: Optionally align assets to a common date set
    if align_dates:
        date_sets = {name: data[return_type].index for name, data in aligned_assets.items()}
        common_dates = sorted(set.intersection(*[set(dates) for dates in date_sets.values()]))
        print(f"\nCommon date range has {len(common_dates)} timestamps.\n")

        # Step 3: Apply intersection to all assets
        for name, data in aligned_assets.items():
            original_len = len(data[return_type])
            aligned_returns = data[return_type].loc[common_dates]
            aligned_len = len(aligned_returns)
            removed = original_len - aligned_len
            threshold = np.quantile(aligned_returns, quantile)

            aligned_assets[name][return_type] = aligned_returns
            aligned_assets[name]['Threshold'] = threshold

            print(f"{name}: {original_len} → {aligned_len} (removed {removed} rows) | Effective start: {effective_start_dates[name].strftime('%Y-%m-%d')}")
    else:
        # Just print stats, no alignment
        print(f"\nAssets analyzed independently (no date alignment).\n")
        for name, data in aligned_assets.items():
            series_len = len(data[return_type])
            print(f"{name}: {series_len} rows | Effective start: {effective_start_dates[name].strftime('%Y-%m-%d')}")

    return aligned_assets


In [ ]:
asset_configs = {
    'BTC':    {'file': 'btcusd_data_20100719_20250516.xlsx', 'date_col': 'Date',       'price_col': 'Price'},
    'BTC_weekend':    {'file': 'btcusd_data_2017-08-17_2025-05-28_binance.xlsx', 'date_col': 'Open Time', 'price_col': 'Close'},
    'ETH':   {'file': 'ethusd_data_20180209_20250516.xlsx',  'date_col': 'Date',       'price_col': 'Price'},
    'ETH_weekend':    {'file': 'ethusd_data_2017-08-17_2025-05-28_binance.xlsx',  'date_col': 'Open Time',  'price_col': 'Close'},
    'S&P500': {'file': 'sp500_data_19480416_20250516.xlsx',   'date_col': 'Date',       'price_col': 'Price'},
    'Gold':   {'file': 'xau_1969-01-31_2025-05-09.xlsx',       'date_col': 'Dates',      'price_col': 'Last Price'},
    'EURUSD': {'file': 'eurusd_1975-01-02_2025-05-16.xlsx',    'date_col': 'Dates',      'price_col': 'Price'}
}

base_path = next(
    path for path in [Path.cwd() / 'Data', *(p / 'Data' for p in Path.cwd().parents)]
    if path.exists()
)
base_path = f"{base_path}/"

assets = load_and_prepare_assets(asset_configs, base_path)

In [ ]:
#-- Updating assets dictionary with threshold values for POT
assets = {
    'BTC': {'Log_Returns': assets['BTC']['Log_Returns'], 'Threshold': 0.10},
    'BTC_weekend': {'Log_Returns': assets['BTC_weekend']['Log_Returns'], 'Threshold': 0.10},
    'ETH': {'Log_Returns': assets['ETH']['Log_Returns'], 'Threshold': 0.10},
    'ETH_weekend': {'Log_Returns': assets['ETH_weekend']['Log_Returns'], 'Threshold': 0.10},
    'S&P500': {'Log_Returns': assets['S&P500']['Log_Returns'], 'Threshold': 0.04},
    'Gold': {'Log_Returns': assets['Gold']['Log_Returns'], 'Threshold': 0.025},
    'EURUSD': {'Log_Returns': assets['EURUSD']['Log_Returns'], 'Threshold': 0.015}
}

### Functions

In [ ]:
# --- Fit distributions ---
def fit_distributions(returns):
    params_norm = norm.fit(returns)
    params_t = t.fit(returns)
    params_lap = laplace.fit(returns)
    return params_norm, params_t, params_lap

# --- 3a. GPD and GEV Fit for Exceedances (EVT) ---
def fit_gpd(returns, threshold, r, threshold_quantile=0.95, tail="right"):
    if tail == "right":
        model = EVA(data=returns)
        model.get_extremes("POT", threshold=threshold, r=r)
        model.fit_model(distribution="genpareto")
        params_gpd = model.model._fit_parameters
        return threshold, params_gpd
    else:
        model = EVA(data=returns)
        model.get_extremes("POT", threshold=threshold, r=r, extremes_type="low")
        model.fit_model(distribution="genpareto")
        params_gpd = model.model._fit_parameters
        return threshold, params_gpd

def fit_gev(returns, block_size, threshold_quantile=0.95):
    model = EVA(data=returns)
    model.get_extremes("BM", block_size=block_size)
    model.fit_model(distribution="genextreme")
    params_gev = model.model._fit_parameters
    return params_gev

# --- Plot Distribution Fit ---
def plot_fit(returns, params_norm, params_t):
    x = np.linspace(min(returns), max(returns), 1000)
    sns.histplot(returns, stat='density', bins=100, label='Empirical')
    plt.plot(x, norm.pdf(x, *params_norm), label='Normal',  color='red')
    plt.plot(x, t.pdf(x, *params_t), label='t-distribution',  color='#e67e22')
    plt.legend()
    plt.title('Distribution Fit')
    plt.show()

In [ ]:
# --- Compile Comparison Table and Plots ---
summary = pd.DataFrame(columns=['Asset', 'Skewness', 'Kurtosis', 'GPD_xi', 'VaR_95', 'ES_95'])

### Value-at-Risk

In [ ]:
#-- Generating assets dictionary for independent EVT analysis
assets_df = build_evt_dataset(
    assets=assets,
    return_type='Log_Returns',
    quantile=0.95,
    start_date='1990-01-01',
    end_date='2025-05-28', # because of Gold
    use_requested_start_date=True,
    align_dates=False
)

In [ ]:
def lopez_loss(y_true, var_forecast):
    """
    Implements the Lopez magnitude loss function for VaR backtesting.

    Parameters:
        y_true (array-like): Array of observed returns/losses.
        var_forecast (array-like): Corresponding array of VaR forecasts.

    Returns:
        float: Average Lopez loss over the period.
        np.ndarray: Array of pointwise Lopez losses.
    """
    y_true = np.asarray(y_true)
    var_forecast = np.asarray(var_forecast)
    # Pointwise Lopez loss as per formula
    loss = np.where(
        y_true <= var_forecast,
        1 + (y_true - var_forecast) ** 2,
        0
    )
    return np.mean(loss), loss

In [ ]:
def compute_rolling_empirical_var(returns, alpha=0.95, compute_es=False):
    #VaR
    empirical_var = returns.quantile(1 - alpha)    
    if compute_es == True:
        # ES
        shortfall_returns = returns[returns <= empirical_var]
        if not shortfall_returns.empty:
            empirical_es = shortfall_returns.mean()
            return empirical_es
        else:
            empirical_es = np.nan
            print("Warning: No returns found below the VaR threshold. ES cannot be calculated.")       
    else:
        return empirical_var

def compute_parametric_vars(returns, alpha=0.95, threshold_quantile=0.95, compute_es = False):
    """
    Compute left-tail Value-at-Risk (VaR) using Normal, t-distribution, and EVT (GPD).

    Parameters:
        returns (pd.Series): Log returns of the asset.
        alpha (float): VaR confidence level (e.g., 0.95).
        threshold_quantile (float): Quantile used as GPD threshold (e.g., 0.95 → 5% left tail).

    Returns:
        Tuple: (Normal VaR, t-distribution VaR, EVT (GPD) VaR)
    """
    
    # Fit Normal distribution
    # VaR
    mu, sigma = norm.fit(returns, method='mle')
    var_norm = norm.ppf(1 - alpha, loc=mu, scale=sigma)
    
    # ES = mu - sigma * φ(z) / (1-alpha), φ = standard normal pdf
    z_alpha = norm.ppf(1-alpha)
    nominator = norm.pdf(z_alpha) * sigma
    denominator = (1-alpha)
    es_norm =  mu - nominator/denominator
    
    
    # Fit t-distribution
    # VaR
    df_t, loc_t, scale_t = t.fit(returns, method='mle')
    var_t = t.ppf(1 - alpha, df=df_t, loc=loc_t, scale=scale_t)

    z_standard = t.ppf(1 - alpha, df=df_t)
    pdf_z_standard = t.pdf(z_standard, df=df_t) # 2. Calculate the PDF of the standard t-distribution at z_standard
    
    # 3. Calculate the core shortfall term
    core_shortfall_term = (pdf_z_standard / (1 - alpha)) * ((df_t + z_standard**2) / (df_t - 1))
    
    # 4. Calculate the Expected Shortfall for returns
    es_t = loc_t - scale_t * core_shortfall_term # The minus sign ensures it's a negative value (representing a loss)
    
    # EVT - GPD for the left tail
    threshold = np.quantile(returns, 1 - threshold_quantile)
    model = EVA(data=returns)
    model.get_extremes("POT", threshold=threshold, extremes_type="low", r="7D")
    model.fit_model(distribution="genpareto")
    params_gpd = model.model._fit_parameters

    gpd_exceedances = len(model.extremes)
    print(f"Number of exceedances used for GPD fitting: {gpd_exceedances}")
    
    Nu = len(model.extremes)  # number of exceedances
    n = len(returns)       # total sample size
    if Nu < 5:
        raise ValueError(f"Too few exceedances ({Nu}) for reliable GPD fit. Consider lowering threshold_quantile.")
    
    # Fit GPD to exceedances
    c = params_gpd['c']
    scale_gpd = params_gpd['scale']
    
    # Compute GPD-based VaR using EVT formula from QRM
    q_tail = (n / Nu) * (1 - alpha) # exceedance probability
    gpd_var = threshold + (scale_gpd / c) * (q_tail**(-c) - 1)

    # Expected shortfall
    first_part = gpd_var/(1-c)
    second_part = (scale_gpd-c*threshold) / (1-c)
    gpd_es = first_part - second_part

    if compute_es == True:
        return es_norm, es_t, gpd_es
    else:
        return var_norm, var_t, gpd_var


In [ ]:
def backtest_var_distributions(returns, asset, quantile=0.95, save_plot=False, save_excel=False,
                                step_quarters=1, train_years=2):
    
    returns = returns.dropna().sort_index()
    quarters = returns.resample('QE').mean().index
    initial_train_end = returns.index.min() + DateOffset(years=train_years)
    quarters = quarters[quarters > initial_train_end]

    results = pd.DataFrame()

    for i in range(len(quarters) - step_quarters):
        train_start = returns.index.min()
        train_end = quarters[i]
        test_start = quarters[i]
        test_end = quarters[i + step_quarters]

        train_data = returns[train_start:train_end]
        test_data = returns[test_start:test_end]

        if len(train_data) < 100 or len(test_data) < 10:
            continue # we skip that window

        var_norm, var_t, var_gpd = compute_parametric_vars(train_data, alpha=quantile)
        empirical_var_test = compute_rolling_empirical_var(test_data, alpha=quantile)

        ## Lopez Test
        # Broadcast each VaR forecast across the test period for Lopez loss calculation
        var_norm_forecast = np.full(len(test_data), var_norm)
        var_t_forecast = np.full(len(test_data), var_t)
        var_gpd_forecast = np.full(len(test_data), var_gpd)

        # Compute Lopez loss for each model
        lopez_losses = {
            'Normal VaR': lopez_loss(test_data, var_norm_forecast)[0],
            't VaR': lopez_loss(test_data, var_t_forecast)[0],
            'GPD VaR': lopez_loss(test_data, var_gpd_forecast)[0]
        }
        
        if not all(np.isfinite(v) for v in lopez_losses.values()):
            continue

        # --- Chossing the best VaR model ---
        zero_loss_models = [name for name, val in lopez_losses.items() if val == 0]
        
        if len(zero_loss_models) == 3:
            best_method = '-'
        elif len(zero_loss_models) == 2:
            best_method = ' and '.join(zero_loss_models)
        else:
            best_method = min(lopez_losses, key=lopez_losses.get)
        # ---------------------------

        row = {
            'Empirical VaR (Test)': empirical_var_test,
            'Normal VaR': var_norm,
            'T-Distribution VaR': var_t,
            'GPD VaR': var_gpd,
            'Average Lopez Loss (Normal)': lopez_losses['Normal VaR'],
            'Average Lopez Loss (t)': lopez_losses['t VaR'],
            'Average Lopez Loss (GPD)': lopez_losses['GPD VaR'],
            'Best VaR Method': best_method,
            'Start': test_start.date(),
            'End': test_end.date()
        }
        results = pd.concat([results, pd.DataFrame([row], index=[test_end.date()])])

    if not results.empty:

        
        overall_best = results['Best VaR Method'].value_counts().idxmax()
        print(f"\nOverall best VaR method: {overall_best}")
    else:
        print("No valid test windows. Check training/test period settings.")

    if save_excel:
        #Excel
        filename = f"{asset}_{step_quarters}_quarterly_var_comparison.xlsx"
        sheet_name = f"{asset}_VaR"
        with pd.ExcelWriter(filename, engine='openpyxl', mode='a' if os.path.exists(filename) else 'w') as writer:
            results.to_excel(writer, sheet_name=sheet_name)
            
        # Latex
        results_rounded = results.copy()
        
        percent_cols = [
            'Empirical VaR (Test)', 'Normal VaR', 'T-Distribution VaR', 'GPD VaR',
            'Average Lopez Loss (Normal)', 'Average Lopez Loss (t)', 'Average Lopez Loss (GPD)'
        ]
        
        for col in percent_cols:
            results_rounded[col] = results_rounded[col].apply(lambda x: f"{x*100:.4f}\\%")
        
        latex_filename = f"{asset}_{step_quarters}_quarterly_var_comparison.tex"
        results_rounded.to_latex(latex_filename, index=False, escape=False, longtable=True, column_format='cccccccccc'
        )

    plot_daily_var_breaches2(results, returns, asset, step_quarters, save_plot)
    return results

In [ ]:
def plot_var_comparison(results_df, asset, save_plot, step_quarters):
    plt.figure(figsize=(10, 5))
    plt.plot(results_df.index, results_df['Empirical VaR (Test)']*100, label='Empirical VaR', color='black', linestyle='-')
    plt.plot(results_df.index, results_df['Normal VaR']*100, label='Normal VaR', color='blue', linestyle='--')
    plt.plot(results_df.index, results_df['T-Distribution VaR']*100, label='t-Distribution VaR', color='green', linestyle='--')
    plt.plot(results_df.index, results_df['GPD VaR']*100, label='GPD VaR', color='red', linestyle='--')
    plt.title(f"VaR Comparison, Quarters used: {step_quarters} - {asset}")
    plt.xlabel("Date")
    plt.ylabel("VaR (%)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    if save_plot:
        filename = f"{asset}_{step_quarters}_quarterly_var_comparison.png"
        plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
def plot_var_comparison2(results_df, asset, save_plot, step_quarters, returns=None):
    fig, ax = plt.subplots(figsize=(10, 5))

    # Plot realized returns (mean return per quarter in %)
    if returns is not None:
        # Resample returns by quarter-end and align with results_df's index
        realized_returns = returns.resample('QE').mean().reindex(results_df.index)
        ax.plot(results_df.index, realized_returns * 100, label='Realized Return', color='black', linestyle='-')

    #ax.plot(results_df.index, results_df['Empirical VaR (Test)']*100, label='Empirical VaR', color='black', linestyle='-')
    ax.plot(results_df.index, results_df['Normal VaR']*100, label='Normal VaR', color='blue', linestyle='--')
    ax.plot(results_df.index, results_df['T-Distribution VaR']*100, label='t-Distribution VaR', color='green', linestyle='--')
    ax.plot(results_df.index, results_df['GPD VaR']*100, label='GPD VaR', color='red', linestyle='--')
    ax.set_title(f"VaR Comparison, Quarters used: {step_quarters} - {asset}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Value (%)")
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    if save_plot:
        filename = f"{asset}_{step_quarters}_quarterly_var_comparison.png"
        plt.savefig(filename, dpi=300)
    plt.show()


In [ ]:
def plot_daily_var_breaches(results_df, returns, asset, step_quarters, save_plot=False):
    """
    Plots daily returns and step-wise VaR predictions (Normal, t, GPD), 
    highlighting breaches where daily return is below any VaR estimate.

    Parameters:
        results_df: DataFrame from backtest_var_distributions (quarterly rows, VaR columns)
        returns: pd.Series of daily returns
        asset: asset name (str)
        step_quarters: int (used in plot title)
        save_plot: bool
    """

    # Prepare a DataFrame with daily VaR forecasts for each method
    var_methods = {
        "Normal VaR": "blue",
        "T-Distribution VaR": "green",
        "GPD VaR": "red"
    }
    daily_vars = pd.DataFrame(index=returns.index, columns=var_methods)

    for col in var_methods:
        # Fill each day with its quarter's VaR value (stepwise constant)
        for end_date, var_val in results_df[col].items():
            start_date = results_df.loc[end_date, 'Start']
            mask = (daily_vars.index > pd.to_datetime(start_date)) & (daily_vars.index <= pd.to_datetime(end_date))
            daily_vars.loc[mask, col] = var_val

    # Cast to float for plotting/math
    daily_vars = daily_vars.astype(float)

    # Find breaches: daily return < ANY VaR forecast (row-wise min)
    breach_mask = (returns < daily_vars.min(axis=1))

    # Plot
    fig, ax = plt.subplots(figsize=(14, 6))

    # Plot daily returns
    ax.plot(returns.index, returns * 100, color='black', linewidth=0.7, label='Daily Return', zorder=2)

    # Plot step-wise VaRs
    for col, color in var_methods.items():
        ax.plot(daily_vars.index, daily_vars[col] * 100, color=color, linestyle='--', label=col, zorder=1)

    # Highlight breaches
    ax.scatter(returns.index[breach_mask], (returns[breach_mask] * 100), 
               color='crimson', s=22, marker='x', label='VaR Breach', zorder=3)

    ax.set_title(f"Daily Returns & Step-wise VaR with Breaches (Quarters: {step_quarters}) - {asset}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Return / VaR (%)")
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    if save_plot:
        filename = f"{asset}_{step_quarters}_daily_var_breaches.png"
        plt.savefig(filename, dpi=300)
    plt.show()


In [ ]:
def plot_daily_var_breaches2(results_df, returns, asset, step_quarters, save_plot=False):
    """
    Plot only the test period, only negative returns, and highlight VaR breaches.
    """
    # 1. Get test period range (from first 'Start' in results_df onward)
    test_start = pd.to_datetime(results_df.iloc[0]['Start'])
    test_end = pd.to_datetime(results_df.index[-1])

    # Filter returns for test period only
    test_returns = returns[(returns.index > test_start) & (returns.index <= test_end)]

    # 2. Only keep negative returns
    neg_returns = test_returns[test_returns < 0]

    # 3. Build daily VaR forecasts (step-wise) for test period only
    var_methods = {
        "Normal VaR": "blue",
        "T-Distribution VaR": "green",
        "GPD VaR": "red"
    }
    daily_vars = pd.DataFrame(index=neg_returns.index, columns=var_methods)
    for col in var_methods:
        for end_date, var_val in results_df[col].items():
            start_date = results_df.loc[end_date, 'Start']
            mask = (daily_vars.index > pd.to_datetime(start_date)) & (daily_vars.index <= pd.to_datetime(end_date))
            daily_vars.loc[mask, col] = var_val
    daily_vars = daily_vars.astype(float)

    # 4. Find breaches: negative daily return < any VaR forecast (row-wise min)
    breach_mask = (neg_returns < daily_vars.min(axis=1)) # since we take the minimum, we will be considering a breach a loss that is < GPD VaR

    # Plot
    fig, ax = plt.subplots(figsize=(14, 6))
    # Plot negative daily returns
    ax.plot(neg_returns.index, neg_returns * 100, color='black', linewidth=0.7, label='Negative Daily Returns', zorder=2)
    # Plot step-wise VaRs
    for col, color in var_methods.items():
        ax.plot(daily_vars.index, daily_vars[col] * 100, color=color, linestyle='--', label=col, zorder=1)
    # Highlight breaches
    ax.scatter(neg_returns.index[breach_mask], (neg_returns[breach_mask] * 100), 
               color='crimson', s=22, marker='x', label='GPD VaR Breach', zorder=3)
    ax.set_title(f"Negative Daily Returns & Step-wise VaR with Breaches (Quarters: {step_quarters}) - {asset}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Return / VaR (%)")
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    if save_plot:
        filename = f"{asset}_{step_quarters}_daily_var_breaches.png"
        plt.savefig(filename, dpi=300)
    plt.show()


### Expected Shortfall

In [ ]:
def backtest_es_distributions(returns, asset, quantile=0.95, save_plot=False, save_excel=False,
                                step_quarters=1, train_years=2):
    returns = returns.dropna().sort_index()
    quarters = returns.resample('QE').mean().index
    initial_train_end = returns.index.min() + pd.DateOffset(years=train_years)
    quarters = quarters[quarters > initial_train_end]

    results = pd.DataFrame()

    for i in range(len(quarters) - step_quarters):
        train_start = returns.index.min()
        train_end = quarters[i]
        test_start = quarters[i]
        test_end = quarters[i + step_quarters]

        train_data = returns[train_start:train_end]
        test_data = returns[test_start:test_end]

        if len(train_data) < 100 or len(test_data) < 10:
            continue # skip if not enough data

        # Compute parametric VaR/ES for this training window
        var_norm, var_t, var_gpd = compute_parametric_vars(train_data, alpha=quantile, compute_es=False)
        es_norm, es_t, gpd_es = compute_parametric_vars(train_data, alpha=quantile, compute_es=True) #  NOTE that we compute the ES here!
        empirical_es_test = compute_rolling_empirical_var(test_data, alpha=quantile, compute_es=True)

        # Broadcast each VaR/ES forecast across the test period
        var_norm_forecast = np.full(len(test_data), var_norm)
        var_t_forecast = np.full(len(test_data), var_t)
        var_gpd_forecast = np.full(len(test_data), var_gpd)
        es_norm_forecast = np.full(len(test_data), es_norm)
        es_t_forecast = np.full(len(test_data), es_t)
        gpd_es_forecast = np.full(len(test_data), gpd_es)

        # Convert test_data to array for vectorized ops
        losses = np.asarray(test_data)

        # ---- Violation-based ES backtest: mean(loss - ES) for violations (loss < VaR) ----
        def mean_es_excess(losses, var_forecast, es_forecast):
            mask = losses < var_forecast   # VaR violation (for losses, more negative)
            excess_losses = abs(losses[mask]) - abs(es_forecast[mask])
            return np.mean(excess_losses) if excess_losses.size > 0 else "No Breach"

        mean_excess_norm = mean_es_excess(losses, var_norm_forecast, es_norm_forecast)
        mean_excess_t = mean_es_excess(losses, var_t_forecast, es_t_forecast)
        mean_excess_gpd = mean_es_excess(losses, var_gpd_forecast, gpd_es_forecast)

        # Select best ES method for this window
        method_excess = {
            "Normal": mean_excess_norm,
            "t": mean_excess_t,
            "GPD": mean_excess_gpd
        }
        def is_no_breach(val):
            # Accepts any string containing "no breach" (case insensitive, ignores leading/trailing whitespace)
            return isinstance(val, str) and "no breach" in val.strip().lower()
        
        no_breach_methods = [m for m, v in method_excess.items() if is_no_breach(v)]
        if no_breach_methods:
            best_es_method = ", ".join(no_breach_methods)
        else:
            eligible = {m: v for m, v in method_excess.items() if isinstance(v, (float, int))}
            if eligible:
                min_excess = min(eligible.values())
                close_methods = [m for m, v in eligible.items() if np.isclose(v, min_excess)]
                best_es_method = ", ".join(close_methods)
            else:
                best_es_method = None


        row = {
            'Empirical ES (Test)': empirical_es_test,
            'Normal ES': es_norm,
            'T-Distribution ES': es_t,
            'GPD ES': gpd_es,
            'Mean Excess (Normal)': mean_excess_norm,
            'Mean Excess (t)': mean_excess_t,
            'Mean Excess (GPD)': mean_excess_gpd,
            'Best ES Method': best_es_method,
            'Start': test_start.date(),
            'End': test_end.date()
        }
        results = pd.concat([results, pd.DataFrame([row], index=[test_end.date()])])
        
        if not results.empty:
            # Deconstruct "Best ES Method" cell if multiple methods are tied (comma-separated)
            all_methods = results['Best ES Method'].dropna().astype(str).tolist()
            flat_methods = []
            for entry in all_methods:
                methods = [m.strip() for m in entry.split(',') if m.strip()]
                flat_methods.extend(methods)
            method_counts = Counter(flat_methods)
            if method_counts:
                overall_best = method_counts.most_common(1)[0][0]
                print(f"\nOverall best ES method: {overall_best}")
                print(f"Method counts: {dict(method_counts)}")
            else:
                overall_best = None
                print("\nNo valid best method found in any window.")
            print('Backtest completed! See mean excess and Best ES Method columns for ES diagnostics.')
        else:
            print("No valid test windows. Check training/test period settings.")


    if save_excel:
        #Excel
        filename = f"{asset}_{step_quarters}_quarterly_es_comparison.xlsx"
        sheet_name = f"{asset}_ES"
        with pd.ExcelWriter(filename, engine='openpyxl', mode='a' if os.path.exists(filename) else 'w') as writer:
            results.to_excel(writer, sheet_name=sheet_name)
            
        # Latex
        # Assuming your DataFrame is 'results'
        results_rounded = results.copy()
        
        percent_cols = [
            'Empirical ES (Test)', 'Normal ES', 'T-Distribution ES', 'GPD ES',
            'Mean Excess (Normal)', 'Mean Excess (t)', 'Mean Excess (GPD)'
        ]
        
        for col in percent_cols:
            results_rounded[col] = results_rounded[col].apply(lambda x: f"{x*100:.4f}\\%" if not isinstance(x, str) else x)
        
        latex_filename = f"{asset}_{step_quarters}_quarterly_es_comparison.tex"
        results_rounded.to_latex(latex_filename, index=False, escape=False, longtable=True, column_format='cccccccccc'
        )

    plot_daily_es_exceedances(results, returns, asset, step_quarters, save_plot)
    return results


In [ ]:
def plot_es_comparison(results_df, asset, save_plot, step_quarters):
    plt.figure(figsize=(10, 5))
    plt.plot(results_df.index, results_df['Empirical ES (Test)']*100, label='Empirical ES', color='black', linestyle='-')
    #plt.plot(results_df.index, assets_df[asset]['Log_Returns']*100, label='Empirical ES', color='black', linestyle='-') 
    plt.plot(results_df.index, results_df['Normal ES']*100, label='Normal ES', color='blue', linestyle='--')
    plt.plot(results_df.index, results_df['T-Distribution ES']*100, label='t-Distribution ES', color='green', linestyle='--')
    plt.plot(results_df.index, results_df['GPD ES']*100, label='GPD ES', color='red', linestyle='--')
    plt.title(f"Expected Shortfall Comparison, Quarters used: {step_quarters} - {asset}")
    plt.xlabel("Date")
    plt.ylabel("ES (%)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    if save_plot:
        filename = f"{asset}_{step_quarters}_quarterly_es_comparison.png"
        plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
def plot_daily_es_exceedances(results_df, returns, asset, step_quarters, save_plot=False):
    """
    Plot only the test period, only negative returns, and highlight ES exceedances (returns below any ES).
    """

    # 1. Get test period range (from first 'Start' in results_df onward)
    test_start = pd.to_datetime(results_df.iloc[0]['Start'])
    test_end = pd.to_datetime(results_df.index[-1])

    # Filter returns for test period only
    test_returns = returns[(returns.index > test_start) & (returns.index <= test_end)]

    # Only keep negative returns
    neg_returns = test_returns[test_returns < 0]

    # 3. Build daily ES forecasts (step-wise) for test period only
    es_methods = {
        "Normal ES": "blue",
        "T-Distribution ES": "green",
        "GPD ES": "red"
    }
    daily_es = pd.DataFrame(index=neg_returns.index, columns=es_methods)
    for col in es_methods:
        for end_date, es_val in results_df[col].items():
            start_date = results_df.loc[end_date, 'Start']
            mask = (daily_es.index > pd.to_datetime(start_date)) & (daily_es.index <= pd.to_datetime(end_date))
            daily_es.loc[mask, col] = es_val
    daily_es = daily_es.astype(float)

    # 4. Find exceedances: negative daily return < max ES forecast (i.e., most conservative/least negative ES)
    exceed_mask = (neg_returns < daily_es.max(axis=1))  # for ES, max is the least negative (most conservative)

    # Plot
    fig, ax = plt.subplots(figsize=(14, 6))
    # Plot negative daily returns
    ax.plot(neg_returns.index, neg_returns * 100, color='black', linewidth=0.7, label='Negative Daily Returns', zorder=2)
    # Plot step-wise ESs
    for col, color in es_methods.items():
        ax.plot(daily_es.index, daily_es[col] * 100, color=color, linestyle='--', label=col, zorder=1)
    # Highlight ES exceedances
    ax.scatter(neg_returns.index[exceed_mask], (neg_returns[exceed_mask] * 100), 
               color='crimson', s=22, marker='x', label='ES Exceedance', zorder=3)
    ax.set_title(f"Negative Daily Returns & Step-wise ES with Exceedances (Quarters: {step_quarters}) - {asset}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Return / ES (%)")
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    if save_plot:
        filename = f"{asset}_{step_quarters}_daily_es_exceedances.png"
        plt.savefig(filename, dpi=300)
    plt.show()


### Plotting the Results

#### VaR

In [ ]:
asset = 'BTC'
backtest_var_distributions(
    returns=assets_df[asset]['Log_Returns'],
    asset=asset,
    quantile=0.95,
    save_plot=False,
    save_excel=False,
    step_quarters=4,
    train_years=2
)

In [ ]:
asset = 'ETH'
backtest_var_distributions(
    returns=assets_df[asset]['Log_Returns'],
    asset=asset,
    quantile=0.95,
    save_plot=False,
    save_excel=False,
    step_quarters=4,
    train_years=2
)

#### Expected Shortfall

In [ ]:
asset = 'BTC'
backtest_es_distributions(
    returns=assets_df[asset]['Log_Returns'],
    asset=asset,
    quantile=0.95,
    save_plot=False,
    save_excel=False,
    step_quarters=1,
    train_years=2
)

In [ ]:
asset = 'ETH'
backtest_es_distributions(
    returns=assets_df[asset]['Log_Returns'],
    asset=asset,
    quantile=0.95,
    save_plot=False,
    save_excel=False,
    step_quarters=1,
    train_years=2
)

**Overall**:
- **Data setup:** This notebook builds independent EVT datasets for BTC, ETH, BTC/ETH weekend series. The usable samples are: BTC = 3,869 rows, ETH = 1,896 rows, BTC_weekend/ETH_weekend = 2,841 rows each.

- **BTC VaR plot/table:** The BTC 95% VaR backtest covers 48 windows from 2013-09-30 to 2025-06-30. GPD VaR is the best method in 44/48 windows, with 4 ties between Normal and GPD. Empirical VaR ranges from -13.66% to -3.15%, while GPD VaR is more conservative, ranging from -16.77% to -10.67%.

- **ETH VaR plot/table:** The ETH 95% VaR backtest covers 18 windows from 2021-03-31 to 2025-06-30. GPD VaR is best in 15/18 windows, with 2 no-breach windows and 1 Normal/GPD tie. Empirical VaR ranges from -9.82% to -4.58%, while GPD VaR ranges from -12.01% to -9.95%.

- **BTC Expected Shortfall plot/table:** The BTC ES backtest covers 51 windows from 2012-12-31 to 2025-03-31. Empirical ES ranges from -34.09% to -2.59%. The t-distribution ES is sometimes extremely conservative/unstable, reaching -359.89%, while GPD ES ranges from -50.35% to -24.95%. The best ES method is mixed: t appears 16 times, all three methods tie 16 times, GPD appears 9 times, and Normal/GPD tie 9 times.

- **ETH Expected Shortfall plot/table:** The ETH ES backtest covers 21 windows from 2020-06-30 to 2025-06-30. GPD is clearly strongest, selected in 15/21 windows, with 5 all-method ties and 1 t-distribution win. Empirical ES ranges from -19.51% to -4.57%, while GPD ES is more conservative, ranging from -31.47% to -18.08%.

- **Overall interpretation:** Across the VaR plots, GPD-based EVT gives the most reliable and conservative risk estimates for BTC and ETH. For ES, GPD is very strong for ETH, while BTC is less clear because the t-distribution and GPD both capture extreme losses, but the t-model can produce unrealistically large ES estimates.